# LAB 03: NHẬN DẠNG TIẾNG NÓI NÂNG CAO VỚI HMM
## BÁO CÁO TOÀN DIỆN 8 QUY TRÌNH (A ĐẾN H)
**Sinh viên thực hiện:** Lê Trọng Huy  
**Mã số sinh viên (MSSV):** 2351260658  
**Môn học:** Xử lý Âm thanh và Tiếng nói (Audio & Speech Processing)  

---

### Mục lục 8 Quy trình:
- **Quy trình A:** Chuẩn hóa tín hiệu & Trích xuất đặc trưng 39 chiều (MFCC + $\Delta$ + $\Delta^2$ + CMVN)
- **Quy trình B:** Khởi tạo cấu trúc HMM Trái-sang-Phải (Left-to-Right Bakis)
- **Quy trình C:** Huấn luyện mô hình cho từng từ bằng thuật toán Baum-Welch (EM)
- **Quy trình D:** Nhận dạng Maximum Likelihood & Căn chỉnh Viterbi Alignment
- **Quy trình E:** So sánh toàn diện với DTW Baseline (Dynamic Time Warping)
- **Quy trình F:** Xây dựng Lexicon Mini & Cơ chế ghép nối Phone HMM
- **Quy trình G:** Mô hình ngôn ngữ Bigram & Tái xếp hạng MAP Re-ranking
- **Quy trình H:** Đánh giá độ bền vững (Robustness) & Ngưỡng từ chối Out-of-Vocabulary (OOV)


In [ ]:
# 0. Khởi tạo môi trường và nạp các thư viện cần thiết
import os
import glob
import time
import numpy as np
import soundfile as sf
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from hmmlearn import hmm
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from IPython.display import Image, display

# Cấu hình danh mục từ lệnh điều khiển tiếng Việt
VOCAB = ["mo", "dong", "len", "xuong", "dung"]
VOCAB_VI = {"mo": "Mở", "dong": "Đóng", "len": "Lên", "xuong": "Xuống", "dung": "Dừng"}

print("Nạp các thư viện thành công!")


---
## QUY TRÌNH A: CHUẨN HÓA VÀ TRÍCH ĐẶC TRƯNG ÂM HỌC
### Các bước thực hiện:
1. Đọc file âm thanh WAV 16kHz mono.
2. Cắt khoảng lặng thừa ở biên (Silence trimming) với ngưỡng năng lượng `top_db=25`.
3. Bộ lọc tiền nhấn (Pre-emphasis filter) làm phẳng độ dốc phổ: $y[n] = x[n] - 0.97 x[n-1]$.
4. Trích xuất 13 hệ số MFCC với cửa sổ Hamming 25ms ($400$ samples), bước nhảy 10ms ($160$ samples), $26$ dải lọc Mel.
5. Tính đạo hàm bậc 1 ($\Delta$ - vận tốc) và bậc 2 ($\Delta^2$ - gia tốc).
6. Ghép nối thành ma trận kích thước $T \times 39$.
7. Áp dụng chuẩn hóa CMVN (Cepstral Mean and Variance Normalization) theo từng utterance.


In [ ]:
def pre_emphasis(signal, alpha=0.97):
    return np.append(signal[0], signal[1:] - alpha * signal[:-1])

def extract_39d_features(audio_path, apply_cmvn=True, sr=16000):
    y, orig_sr = sf.read(audio_path)
    if y.ndim > 1:
        y = np.mean(y, axis=1)
    if orig_sr != sr:
        y = librosa.resample(y, orig_sr=orig_sr, target_sr=sr)
    
    # 1. Cắt khoảng lặng thừa
    y_trim, _ = librosa.effects.trim(y, top_db=25)
    if len(y_trim) < 400:
        y_trim = y
        
    # 2. Bộ lọc tiền nhấn
    y_emp = pre_emphasis(y_trim, alpha=0.97)
    
    # 3. Trích xuất 13 MFCCs
    mfcc = librosa.feature.mfcc(
        y=y_emp, sr=sr, n_mfcc=13, n_fft=512,
        win_length=400, hop_length=160, n_mels=26, window="hamming"
    )
    
    # 4. Tính Delta và Delta-Delta
    delta1 = librosa.feature.delta(mfcc, order=1, width=5)
    delta2 = librosa.feature.delta(mfcc, order=2, width=5)
    
    # 5. Ghép nối thành ma trận T x 39
    feat_39 = np.concatenate([mfcc, delta1, delta2], axis=0).T
    
    # 6. CMVN (Chuẩn hóa trung bình và phương sai)
    if apply_cmvn and feat_39.shape[0] > 1:
        mean = np.mean(feat_39, axis=0, keepdims=True)
        std = np.std(feat_39, axis=0, keepdims=True)
        std[std < 1e-6] = 1.0
        feat_39 = (feat_39 - mean) / std
        
    return feat_39, mfcc.T, delta1.T, delta2.T

print("Hàm trích xuất đặc trưng 39D sẵn sàng!")


In [ ]:
# Hiển thị trực quan Heatmaps của 2 từ: 'Mở' và 'Đóng'
print("1. Heatmap 13 MFCC / 13 Delta / 13 Delta-Delta của từ 'Mở':")
display(Image("figures/heatmap_features_mo.png"))

print("2. Heatmap 13 MFCC / 13 Delta / 13 Delta-Delta của từ 'Đóng':")
display(Image("figures/heatmap_features_dong.png"))


### Phân tích khác biệt âm học:
- **Từ 'Mở' (/m/ - /ɤ̌/):** Bắt đầu bằng âm mũi khép môi /m/ với năng lượng dải tần thấp tập trung quanh các hệ số $c_0 - c_3$, sau đó chuyển tiếp mượt mà sang nguyên âm mở /ɤ/ với năng lượng phổ trải rộng. Các hệ số Delta biến thiên ổn định.
- **Từ 'Đóng' (/ɗ/ - /awŋm/):** Bắt đầu bằng phụ âm tắc thanh môn /ɗ/ tạo xung đột biến rõ rệt trên dải $\Delta$ và $\Delta^2$, sau đó chuyển sang nguyên âm đôi và kết thúc bằng âm mũi ngạc mềm /ŋm/ với năng lượng dồn về cuối.


---
## QUY TRÌNH B: KHỞI TẠO HMM TRÁI-SANG-PHẢI (LEFT-TO-RIGHT BAKIS)
### Yêu cầu cấu hình:
- Chọn $N = 5$ trạng thái cho mỗi từ.
- Xác suất ban đầu: $\pi = [1, 0, 0, 0, 0]$ (chỉ bắt đầu ở trạng thái 0).
- Ma trận chuyển trạng thái Bakis:
  + $a_{ii} \approx 0.7$ (Self-loop)
  + $a_{i, i+1} \approx 0.3$ (Chuyển sang trạng thái kế tiếp)
  + Trạng thái kết thúc: $a_{nn} = 1.0$ (Absorbing state).
  + Không có chuyển lùi ($a_{ij} = 0$ khi $j < i$) và không nhảy cóc ($a_{ij} = 0$ khi $j > i + 1$).
- Dùng Gaussian với ma trận hiệp phương sai đường chéo (`diag`) và variance floor $10^{-3}$.


In [ ]:
def init_left_to_right_hmm(n_states=5, self_loop=0.7, covar_type="diag"):
    model = hmm.GaussianHMM(
        n_components=n_states,
        covariance_type=covar_type,
        n_iter=60,
        min_covar=1e-3,
        random_state=42,
        init_params="stmc",
        params="stmc"
    )
    
    startprob = np.zeros(n_states)
    startprob[0] = 1.0
    
    transmat = np.zeros((n_states, n_states))
    for i in range(n_states):
        if i == n_states - 1:
            transmat[i, i] = 1.0
        else:
            transmat[i, i] = self_loop
            transmat[i, i + 1] = 1.0 - self_loop
            
    # Thêm pseudo-count rất nhỏ chống lỗi zero-sum row
    transmat = transmat + 1e-6
    transmat = transmat / transmat.sum(axis=1, keepdims=True)
    
    model.startprob_ = startprob
    model.transmat_ = transmat
    return model

m_demo = init_left_to_right_hmm(n_states=5, self_loop=0.7)
print("Ma trận chuyển trạng thái khởi tạo Left-to-Right:")
print(np.round(m_demo.transmat_, 3))


---
## QUY TRÌNH C: HUẤN LUYỆN HMM VỚI THUẬT TOÁN BAUM-WELCH (EM)
### Các bước thực hiện:
- Nối các utterance train cùng nhãn thành `X_concat` và truyền mảng `lengths` vào `fit()`.
- Thuật toán Baum-Welch (EM) tự động tối ưu hóa $\mu, \Sigma, A$ theo tiêu chuẩn cực đại hợp lý.
- Kiểm tra tính hội tụ và ma trận `transmat_` sau khi huấn luyện.


In [ ]:
# Nạp dữ liệu Train của Speaker 1
train_data = {w: [] for w in VOCAB}
test_data = {w: [] for w in VOCAB}
mismatch_data = {w: [] for w in VOCAB}

for w in VOCAB:
    for fpath in glob.glob(f"dataset/speaker1/train/{w}/*.wav"):
        feat, _, _, _ = extract_39d_features(fpath, apply_cmvn=True)
        train_data[w].append(feat)
    for fpath in glob.glob(f"dataset/speaker1/test/{w}/*.wav"):
        feat, _, _, _ = extract_39d_features(fpath, apply_cmvn=True)
        test_data[w].append(feat)
    for fpath in glob.glob(f"dataset/speaker2/test/{w}/*.wav"):
        feat, _, _, _ = extract_39d_features(fpath, apply_cmvn=True)
        mismatch_data[w].append(feat)

print(f"Nạp dữ liệu: {sum(len(v) for v in train_data.values())} train, {sum(len(v) for v in test_data.values())} test.")

# Huấn luyện mô hình cho 5 từ
hmm_models = {}
for w in VOCAB:
    X_concat = np.concatenate(train_data[w], axis=0)
    lengths = [len(f) for f in train_data[w]]
    model = init_left_to_right_hmm(n_states=5, self_loop=0.7)
    model.fit(X_concat, lengths=lengths)
    hmm_models[w] = model
    print(f"Huấn luyện thành công HMM cho từ '{VOCAB_VI[w]}' (converged: {model.monitor_.converged})")


---
## QUY TRÌNH D: NHẬN DẠNG MAXIMUM LIKELIHOOD VÀ VITERBI ALIGNMENT
### Các bước thực hiện:
1. Tính $\log P(X \mid \lambda_w)$ cho mọi từ $w \in W$ và chọn $\hat{w} = \arg\max_w \log P(X \mid \lambda_w)$.
2. Dùng `decode(..., algorithm="viterbi")` lấy chuỗi trạng thái ẩn tối ưu.
3. Vẽ đồ thị bậc thang `plt.step` thể hiện chuỗi trạng thái theo frame và liên hệ với các pha ngữ âm.


In [ ]:
# 1. Bảng điểm Log-Likelihood trên 1 mẫu kiểm thử
sample_test = test_data["mo"][0]
scores = {w: hmm_models[w].score(sample_test) for w in VOCAB}
best_pred = max(scores, key=scores.get)

print("BẢNG ĐIỂM LOG-LIKELIHOOD MINH HỌA (Mẫu test thực tế: 'Mở'):")
print(f"{'Mô hình HMM':<15} | {'Log-Likelihood':<18} | {'Normalized (LL/T)':<18} | {'Quyết định'}")
print("-" * 65)
for w in VOCAB:
    s = scores[w]
    sn = s / len(sample_test)
    marker = " <== [CHỌN]" if w == best_pred else ""
    print(f"{VOCAB_VI[w]:<15} | {s:>18.2f} | {sn:>18.2f} |{marker}")


In [ ]:
# 2. Hiển thị đồ thị Viterbi Alignment Path
print("Viterbi Path - Từ 'Mở':")
display(Image("figures/viterbi_path_mo.png"))

print("Viterbi Path - Từ 'Đóng':")
display(Image("figures/viterbi_path_dong.png"))


### Nhận xét Viterbi Path:
- Đường căn chỉnh diễn tiến đơn điệu từ State 0 sang State 4 (không lùi, đúng ràng buộc Bakis).
- State 2 chiếm số lượng frame lớn nhất ($20-25$ frames), tương ứng với pha nguyên âm ổn định nhất của từ.


---
## QUY TRÌNH E: SO SÁNH VỚI DTW BASELINE (DYNAMIC TIME WARPING)
### Đối sánh giữa Template Matching (DTW 1-NN) và Statistical Modeling (HMM):


In [ ]:
display(Image("figures/cm_dtw.png"))
display(Image("figures/dtw_vs_hmm_comparison.png"))


### Bảng Kết Quả Đối Soánh Toàn Diện:
| Tiêu chí | DTW Baseline (1-NN) | HMM (Left-to-Right Gaussian) | Đánh giá ưu thế |
|:---|:---:|:---:|:---|
| **Độ chính xác (Clean Test)** | $100.00\%$ | **$100.00\%$** | Ngang nhau trên tập sạch |
| **Thời gian suy luận / mẫu** | $143.28\text{ ms}$ | **$1.89\text{ ms}$** | **HMM nhanh hơn gần 75 lần!** |
| **Độ phức tạp suy luận** | $\mathcal{O}(M \cdot T_1 T_2 \cdot D)$ (Tăng tuyến tính theo số mẫu train $M$) | $\mathcal{O}(W \cdot N^2 T \cdot D)$ (Độc lập với số mẫu train) | **HMM vượt trội cho Real-time** |
| **Dung lượng lưu trữ mô hình** | Rất lớn (phải lưu toàn bộ template thô) | Rất nhỏ (chỉ lưu các ma trận số thực $\mu, \Sigma, A$) | **HMM nén tri thức thống kê** |


---
## QUY TRÌNH F: LEXICON MINI VÀ GHÉP NỐI PHONE HMM
### Đọc file từ điển phát âm `lexicon.txt`:


In [ ]:
def parse_lexicon(path="lexicon.txt"):
    lex = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                lex[parts[0]] = parts[1:]
    return lex

lexicon = parse_lexicon()
print("Từ điển Lexicon phát âm mini:")
for w, phones in lexicon.items():
    print(f"  {w:<6} -> {' '.join(phones)}")


### Cơ chế ghép nối Phone HMM thành Word HMM:
- Mỗi âm vị (Phone) được mô hình hóa bởi một HMM nhỏ (thường 3 trạng thái).
- Khi tạo mô hình cho từ, trạng thái kết thúc của Phone $k$ được nối với trạng thái bắt đầu của Phone $k+1$ bằng bước chuyển null xác suất $1.0$.
- Cho phép nhận dạng tập từ vựng mở rộng (Large Vocabulary CSR) mà không cần thu âm riêng từng từ.


---
## QUY TRÌNH G: BIGRAM LANGUAGE MODEL VÀ MAP RE-RANKING
### Công thức tái xếp hạng MAP:
$$\text{Total Score} = \frac{\text{Acoustic Score}}{T} + \lambda_{LM} \cdot \log P(w \mid w_{prev})$$
Với xác suất Bigram áp dụng Add-$0.5$ Smoothing:
$$P(w \mid w_{prev}) = \frac{C(w_{prev}, w) + 0.5}{C(w_{prev}) + 0.5 \cdot |V|}$$


In [ ]:
class BigramLM:
    def __init__(self, corpus_path="corpus_commands.txt", k=0.5):
        self.k = k
        self.unigrams = {}
        self.bigrams = {}
        self.vocab = set()
        with open(corpus_path, "r", encoding="utf-8") as f:
            for line in f:
                words = line.strip().lower().split()
                for w in words:
                    self.vocab.add(w)
                    self.unigrams[w] = self.unigrams.get(w, 0) + 1
                for w1, w2 in zip(words[:-1], words[1:]):
                    bg = (w1, w2)
                    self.bigrams[bg] = self.bigrams.get(bg, 0) + 1
                    
    def log_prob(self, prev, curr):
        V = len(self.vocab)
        c_prev = self.unigrams.get(prev.lower(), 0)
        c_bg = self.bigrams.get((prev.lower(), curr.lower()), 0)
        return float(np.log((c_bg + self.k) / (c_prev + self.k * V)))

lm = BigramLM()
print("Log-prob bigram P('cửa' | 'mở') =", round(lm.log_prob("mở", "cửa"), 3))
print("Log-prob bigram P('xe' | 'dừng')  =", round(lm.log_prob("dừng", "xe"), 3))


### Phân tích hiện tượng Re-ranking:
- **Khi nào LM sửa lỗi:** Khi hai ứng viên âm học có điểm số xấp xỉ nhau (ví dụ phát âm dính chữ hoặc có tiếng ồn), mô hình ngôn ngữ cung cấp tiên nghiệm ngữ nghĩa (ngữ cảnh "cửa" thì đi liền với "mở/đóng"), giúp lật ngược quyết định về từ đúng.
- **Khi nào LM lấn át âm học:** Khi trọng số $\lambda_{LM}$ quá lớn ($\ge 1.0$), xác suất ngữ pháp áp đảo hoàn toàn tín hiệu âm thanh thực tế từ micro, dẫn đến nhận dạng sai lệch theo chủ quan của ngữ pháp.


---
## QUY TRÌNH H: ĐÁNH GIÁ ĐỘ BỀN VỮNG (ROBUSTNESS) VÀ REJECT THRESHOLD OOV
### Kết quả kiểm thử trên Clean Test, Mismatch Test và Phân bố OOV:


In [ ]:
display(Image("figures/cm_clean_hmm.png"))
display(Image("figures/cm_mismatch_hmm.png"))
display(Image("figures/oov_score_distribution.png"))


### Bảng Kết Quả Độ Bền Vững:
- **Độ chính xác Clean Test (Cùng người nói):** **$100.00\%$**
- **Độ chính xác Mismatch Test (Khác người nói - Speaker 2):** **$70.00\%$** (Sụt giảm $30\%$ do sự khác biệt về $F_0$ và khoảng cách formant giữa nam và nữ).
- **Ngưỡng đề xuất từ chối OOV (Reject Threshold):** $\theta = -76.36$.
  + Nếu $\max_w \frac{\log P(X \mid \lambda_w)}{T} \ge \theta$: Chấp nhận kết quả.
  + Nếu $\max_w \frac{\log P(X \mid \lambda_w)}{T} < \theta$: Bác bỏ nhận dạng (Từ chối OOV).
